## Tensile test data

#### <p style="text-align: right;"> &#9989; **put your name here** </p>


---
Here, we will use ANN to do regressions of stress-strain curves. 

- Download the stress-strain curves at 3 different temperatures.
    - `https://raw.githubusercontent.com/huichiayu/cmse_202_802/main/data/stress_strain_T000.csv`
    - `https://raw.githubusercontent.com/huichiayu/cmse_202_802/main/data/stress_strain_T400.csv`
    - `https://raw.githubusercontent.com/huichiayu/cmse_202_802/main/data/stress_strain_T600.csv`

- Load and plot the data.

In [ ]:
# !Curl -O  https://raw.githubusercontent.com/huichiayu/cmse_202_802/main/data/stress_strain_T000.csv
# !Curl -O  https://raw.githubusercontent.com/huichiayu/cmse_202_802/main/data/stress_strain_T400.csv
# !Curl -O  https://raw.githubusercontent.com/huichiayu/cmse_202_802/main/data/stress_strain_T600.csv

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

data_000=pd.read_csv("stress_strain_T000.csv")
data_400=pd.read_csv("stress_strain_T400.csv")
data_600=pd.read_csv("stress_strain_T600.csv")
data_000.shape

plt.scatter(data_000["strain"],data_000["stress"],label='0 C')
plt.scatter(data_400["strain"],data_400["stress"],label='400 C')
plt.scatter(data_600["strain"],data_600["stress"],label='600 C')

plt.xlabel('strain')
plt.ylabel('stress')
plt.grid(True)
plt.legend()
plt.title("tensile test at different temperatures")

### ANN model in `sklearn` libeary

Here, we will use `MLPRegressor` in `sklearn` library. First, let's build a ANN model for regression of the 0-C dataset. 
- $X$ is the input data: strain.
- $y$ is the output data: stress.
- Use `StandardScaler` and `fit_transform` to scale the data.

In [ ]:
import numpy as np

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Your data
X = data_000["strain"].values
y = data_000["stress"].values

# sklearn needs X to be 2D
X = X.reshape(-1, 1)

# ======================================================
# Scale input
# ======================================================
scaler = StandardScaler()

# fit scaler and transform X
X_scaled = scaler.fit_transform(X)

# ======================================================
# Create ANN model
# ======================================================
model = MLPRegressor(
    hidden_layer_sizes=(64,),
    activation="tanh",
    solver="adam",
    learning_rate_init=1e-3,
    max_iter=50000,
    random_state=42)

# train
model.fit(X_scaled, y)

# prediction
strain_smooth = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)
strain_smooth_scaled = scaler.transform(strain_smooth)
stress_pred = model.predict(strain_smooth_scaled)


# metrics
y_pred = model.predict(X_scaled)

print("R² =", r2_score(y, y_pred))
print("MAE =", mean_absolute_error(y, y_pred))
print("RMSE =", np.sqrt(mean_squared_error(y, y_pred)))

# plot
plt.figure(figsize=(6,4))
plt.scatter(X, y, label="Data")
plt.plot(strain_smooth, stress_pred, 'r', label="ANN regression")
plt.xlabel("Strain")
plt.ylabel("Stress")
plt.legend()
plt.grid(True)
plt.show()

print(model.n_iter_)
print(model.loss_)

There are several input arguments in the `MLRegressor`. For instant, choice of activation function (`"tanh"`), sovler (`"adam"`), numbers of neurons and hidden layers, etc.

&#9989;  **DO THIS --** 
1. Do a search for different activation functions: e.g., `identity`, `logistic`, `relu`. What are the difference?
2. Do a search for different solvers. What are the difference?
3. Try increase or decrease numbers of neurons and layers. For example, 64 neurons with 1, 2, 3, layers. Compare which one gives you the best result.

Your answer:


--- 
### Full regression including temperature effects. 

Now, we want to include temperatures as another input feature. In this case, each data point has 2 input fearures: strain and temperature. $X$ will become a Nx2 array, where the first column is still the strain but the second column is temperature.

In [ ]:
strain_000 = data_000["strain"].values
stress_000 = data_000["stress"].values
temp_000 = np.full_like(strain_000, 0.0)

strain_400 = data_400["strain"].values
stress_400 = data_400["stress"].values
temp_400 = np.full_like(strain_400, 400.0)

strain_600 = data_600["strain"].values
stress_600 = data_600["stress"].values
temp_600 = np.full_like(strain_600, 600.0)

# Combine data
strain_all = np.concatenate([strain_000, strain_400, strain_600])
temp_all   = np.concatenate([temp_000, temp_400, temp_600])
stress_all = np.concatenate([stress_000, stress_400, stress_600])

# Input features: [strain, temperature]
X = np.column_stack([strain_all, temp_all])
y = stress_all 
print(X.shape)

# Scale X and y
X_scaler = StandardScaler()
X_scaled = X_scaler.fit_transform(X)

# ANN model
model = MLPRegressor(
    hidden_layer_sizes=(64, 64),
    activation="tanh",
    solver="adam",
    max_iter=10000,
    random_state=42
)

# Train
model.fit(X_scaled, y)

In [ ]:
# Prediction helper
def predict_stress(strain, T):
    strain = np.asarray(strain)

    X_new = np.column_stack([strain, T])
    X_new_scaled = X_scaler.transform(X_new)

    stress_pred = model.predict(X_new_scaled)

    return stress_pred

### Predict stress at different strain and temperatures

In [ ]:
# grid of strain values
strain_plot = np.linspace(0, 0.02, 300)
# temprature
T = np.linspace(300, 800, 300)

stress_pred   = predict_stress(strain_plot, T)


plt.figure(figsize=(6,4))

plt.scatter(data_000["strain"],data_000["stress"],label='0 C')
plt.scatter(data_400["strain"],data_400["stress"],label='400 C')
plt.scatter(data_600["strain"],data_600["stress"],label='600 C')

plt.plot(strain_plot, stress_pred, 'r', linewidth=3, label="varying temp ANN")

plt.xlabel('strain')
plt.ylabel('stress')
plt.grid(True)
plt.legend()
plt.title("tensile test at different temperatures")
plt.show()

&#9989;  **DO THIS --** 
ANN model can extrapolate to the regions outside the data used for training. Do you think the predictions trustable? Why?

Your answer:


# Please upload your completed notebook file via this [LINK](https://www.dropbox.com/request/33jmz3may9u0znvhfx5l). 